# TechStore AI — Project Report Notebook

Notebook này là bản rút gọn theo phong cách báo cáo đồ án. Mục tiêu là nắm nhanh quy mô dữ liệu, chất lượng dữ liệu, hành vi mua hàng và bức tranh tổng quan các mô hình AI của TechStore.

**Điểm nhấn**
- Chỉ giữ các phần quan trọng nhất
- Tối ưu để dễ trình bày và dễ đọc
- Tự động nạp dữ liệu mới nhất từ project


## 1. Thiết lập môi trường
Import thư viện và cấu hình hiển thị gọn gàng.

In [12]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
CSV_DIR = DATA_DIR / 'csv'
MODELS_DIR = ROOT / 'models'
print('Environment ready')

Environment ready


## 2. Nạp dữ liệu
Đọc sản phẩm, người dùng, đơn hàng và các tài nguyên mô hình đã huấn luyện.

In [13]:
products = json.loads((DATA_DIR / 'products.json').read_text(encoding='utf-8'))
orders = pd.read_csv(CSV_DIR / 'synthetic_400k.csv')
orders['date'] = pd.to_datetime(orders['date'])

users_path = CSV_DIR / 'users.csv'
users = pd.read_csv(users_path) if users_path.exists() else pd.DataFrame()

vectors_path = DATA_DIR / 'product_vectors.json'
vectors_meta = json.loads(vectors_path.read_text(encoding='utf-8')) if vectors_path.exists() else {}

print(f"Products: {len(products):,}")
print(f"Orders:   {len(orders):,}")
print(f"Users:    {orders['user_id'].nunique():,}")
print(f"Period:   {orders['date'].min().date()} -> {orders['date'].max().date()}")

Products: 20,000
Orders:   400,000
Users:    49,985
Period:   2023-01-01 -> 2024-12-31


## 3. Tổng quan dữ liệu
Xem nhanh quy mô và một vài chỉ số mô tả quan trọng.

In [14]:
summary = pd.DataFrame({
    'Metric': ['Products', 'Orders', 'Users', 'Categories', 'Brands', 'Avg order rating'],
    'Value': [
        len(products),
        len(orders),
        orders['user_id'].nunique(),
        pd.Series([p.get('category') for p in products]).nunique(),
        pd.Series([p.get('brand') for p in products]).nunique(),
        round(float(orders['rating_order'].mean()), 2),
    ]
})
summary

,Metric,Value
0,Products,20000.00
1,Orders,400000.00
2,Users,49985.00
3,Categories,8.00
4,Brands,187.00
5,Avg order rating,4.05


## 4. Chất lượng dữ liệu
Kiểm tra thiếu dữ liệu, trùng lặp và cấu trúc cột.

In [15]:
quality = pd.DataFrame({
    'Dataset': ['products', 'orders', 'users'],
    'Rows': [len(products), len(orders), len(users)],
    'Missing cells': [
        pd.DataFrame(products).isna().sum().sum(),
        int(orders.isna().sum().sum()),
        int(users.isna().sum().sum()) if not users.empty else 0,
    ],
    'Duplicate rows': [
        pd.DataFrame(products).duplicated().sum(),
        int(orders.duplicated().sum()),
        int(users.duplicated().sum()) if not users.empty else 0,
    ]
})
quality

TypeError: unhashable type: 'dict'

## 5. Cấu trúc sản phẩm
Phân tích danh mục, thương hiệu, rating và giá bán.

In [ ]:
products_df = pd.DataFrame(products)
products_df['price_million'] = products_df['price'] / 1e6

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

products_df['category'].value_counts().head(8).plot(kind='bar', ax=axes[0], color='#4C78A8')
axes[0].set_title('Top categories')
axes[0].set_xlabel('')

products_df['brand'].value_counts().head(8).plot(kind='bar', ax=axes[1], color='#F58518')
axes[1].set_title('Top brands')
axes[1].set_xlabel('')

sns.histplot(products_df['price_million'], bins=30, ax=axes[2], color='#54A24B')
axes[2].set_title('Price distribution (million VND)')

plt.tight_layout()
plt.show()

## 6. Hành vi mua hàng
Quan sát doanh thu, rating, trạng thái đơn và xu hướng theo thời gian.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

orders['month'] = orders['date'].dt.to_period('M').astype(str)
orders.groupby('month').size().tail(12).plot(ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Orders by month')

orders['rating_order'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 1], color='#F58518')
axes[0, 1].set_title('Order rating distribution')

orders['order_status'].value_counts().head(8).plot(kind='bar', ax=axes[1, 0], color='#54A24B')
axes[1, 0].set_title('Order status')

orders.groupby('payment_method').size().sort_values(ascending=False).head(8).plot(kind='bar', ax=axes[1, 1], color='#E45756')
axes[1, 1].set_title('Payment methods')

plt.tight_layout()
plt.show()

## 7. Đánh giá mô hình rating
Tóm tắt mô hình dự đoán rating đã huấn luyện và chỉ số chất lượng.

In [ ]:
rating_model_path = MODELS_DIR / 'm7_rating_predictor.pkl'
if rating_model_path.exists():
    import pickle
    model_pack = pickle.loads(rating_model_path.read_bytes())
    model_summary = pd.DataFrame({
        'Item': ['Model name', 'MAE', 'RMSE', 'Features'],
        'Value': [
            model_pack.get('model_name', 'unknown'),
            round(float(model_pack.get('mae', 0)), 4),
            round(float(model_pack.get('rmse', 0)), 4),
            len(model_pack.get('feature_names', [])),
        ]
    })
    display(model_summary)
else:
    print('Rating model not found')

## 8. Tổng quan các model AI
Xem nhanh toàn bộ model phục vụ gợi ý và truy hồi trong hệ thống.

In [ ]:
models_overview = pd.DataFrame([
    ['Semantic vector search', 'product_vectors.json', 'Retrieval theo embedding'],
    ['TF-IDF content based', 'm2_tfidf_topk.json', 'Gợi ý theo nội dung'],
    ['Item-item CF', 'm3_item_cf_topk.json', 'Gợi ý theo đồng mua'],
    ['SVD factorization', 'm4_*.npy + m4_svd_meta.json', 'Latent factor user-item'],
    ['Popularity / trending', 'm5_popularity.json', 'Xếp hạng theo xu hướng'],
    ['User clustering', 'm6_clusters.json', 'Phân khúc khách hàng'],
    ['Rating predictor', 'm7_rating_predictor.pkl', 'Dự đoán rating đơn hàng'],
], columns=['Model', 'Artifact', 'Purpose'])
models_overview

## 9. Kết luận
Bộ dữ liệu hiện tại đủ lớn để phục vụ phân tích, huấn luyện mô hình và demo hệ thống tư vấn e-commerce. Notebook này giữ lại phần cốt lõi để trình bày gọn, rõ và chuyên nghiệp.